# Canonical CRSP Daily Source — Point-in-Time Common Stocks

This notebook is the single bridge between the immutable Shumway-adjusted CRSP daily panel and the empirical universe-construction pipeline.

It performs three tasks only:

1. identifies CRSP ordinary common stocks using the CIZ characteristics corresponding to legacy share codes 10 and 11;
2. retains the complete daily path of every `PERMNO` that is observed as an ordinary common stock at least once, including later holding-period and delisting observations;
3. projects the panel to the columns required downstream and publishes explicit point-in-time eligibility flags.

The full RAW file remains immutable. Universe breakpoints, the 60-month history rule, and Top-100 rankings remain the responsibility of `01_build_pit_big_small_caps.ipynb`.

## 1. Execution and memory policy

The source contains more than 70 million observations. Every large operation below is therefore lazy and streamed to Parquet. The notebook never calls `read_parquet`, `to_pandas`, or `collect` on the daily panel. The only materialised cross-sectional object is the set of eligible `PERMNO`s, whose size is small relative to the daily panel.

The output is first written to a temporary file and atomically promoted only after the integrity assertions pass. An existing output is reused only when its source metadata and configuration digest match the signed manifest.

In [ ]:
# 1. CONFIGURATION
from __future__ import annotations

from datetime import date
from hashlib import sha256
import json
from pathlib import Path
from typing import Any

import polars as pl
import pyarrow.parquet as pq

ROOT = Path.cwd().resolve()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SOURCE = ROOT / 'data' / 'crsp_daily_shumway_delisting_processed.parquet'
OUTPUT = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.parquet'
MANIFEST = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.manifest.json'

START_DATE = date(1990, 1, 1)
END_DATE = date(2025, 12, 31)
INVESTABLE_EXCHANGES = ['N', 'A', 'Q']  # NYSE, AMEX, NASDAQ
COMMON_ISSUER_TYPES = ['ACOR', 'CORP']

FORCE_RECOMPUTE = False
COMPUTE_SOURCE_SHA256_ON_BUILD = True   # sequential I/O; bounded memory
VERIFY_SOURCE_SHA256_ON_RELOAD = False  # metadata guard is used by default
ROW_GROUP_SIZE = 250_000
COMPRESSION = 'zstd'
COMPRESSION_LEVEL = 3

assert SOURCE.exists(), SOURCE
assert SOURCE != OUTPUT
print(f'Source : {SOURCE} ({SOURCE.stat().st_size / 2**30:.2f} GiB)')
print(f'Output : {OUTPUT}')
print(f'Polars : {pl.__version__}')

## 2. Frozen source and output contracts

The classification follows the [WRDS CRSP CIZ-to-SIZ mapping](https://wrds-www.wharton.upenn.edu/pages/wrds-research/macros/crsp-ciztosiz-macro/) of legacy ordinary-common-share codes 10 and 11:

- `SecurityType == EQTY`;
- `SecuritySubType == COM`;
- `ShareType == NS`;
- `USIncFlg == Y`;
- `IssuerType` is `ACOR` or `CORP`.

This rule excludes ADRs, REITs, closed-end funds, ETFs, and foreign ordinary shares from formation eligibility. `ConditionalType == RW`, `TradingStatusFlg == A`, and an admitted exchange are imposed separately because they describe whether an otherwise valid common stock is eligible at a particular date.

Classification columns are retained in the compact output. This makes every future formation decision auditable without reopening the 4 GiB source.

In [ ]:
ESSENTIAL_COLUMNS = [
    'PERMNO', 'PERMCO', 'CUSIP', 'Ticker', 'DlyCalDt',
    'DlyRet', 'DlyRetx', 'DlyRetI',
    'DlyPrc', 'DlyClose', 'DlyHigh', 'DlyLow', 'DlyOpen',
    'DlyCap', 'DlyVol', 'DlyPrcVol', 'ShrOut',
    'vwretd', 'vwretx', 'ewretd', 'sprtrn',
    'PrimaryExch', 'SICCD', 'NAICS',
    'DelistingDt', 'DelRet', 'DelReasonType', 'delist_category',
]

CLASSIFICATION_COLUMNS = [
    'ConditionalType', 'TradingStatusFlg', 'SecurityActiveFlg',
    'SecurityType', 'SecuritySubType', 'ShareType',
    'USIncFlg', 'IssuerType', 'ShrAdrFlg',
]

FLAG_COLUMNS = [
    'is_common_stock_10_11',
    'is_regular_active',
    'is_investable_exchange',
    'is_formation_eligible',
    'is_adr_diagnostic',
    'is_reit_diagnostic',
    'is_fund_diagnostic',
]

SOURCE_COLUMNS = list(dict.fromkeys(ESSENTIAL_COLUMNS + CLASSIFICATION_COLUMNS))
OUTPUT_COLUMNS = SOURCE_COLUMNS + FLAG_COLUMNS

assert len(ESSENTIAL_COLUMNS) == 28
assert len(OUTPUT_COLUMNS) == len(set(OUTPUT_COLUMNS))

source_file = pq.ParquetFile(SOURCE)
source_meta = source_file.metadata
source_arrow_schema = source_file.schema_arrow
missing = sorted(set(SOURCE_COLUMNS) - set(source_arrow_schema.names))
assert not missing, f'Missing source columns: {missing}'

schema_payload = [(field.name, str(field.type)) for field in source_arrow_schema]
source_schema_sha256 = sha256(
    json.dumps(schema_payload, separators=(',', ':')).encode('utf-8')
).hexdigest()

source_signature = {
    'path': str(SOURCE.relative_to(ROOT)),
    'size_bytes': SOURCE.stat().st_size,
    'mtime_ns': SOURCE.stat().st_mtime_ns,
    'rows': source_meta.num_rows,
    'row_groups': source_meta.num_row_groups,
    'schema_sha256': source_schema_sha256,
}

config_payload = {
    'schema_version': 'crsp-common-stock-pit-source-v1',
    'start_date': START_DATE.isoformat(),
    'end_date': END_DATE.isoformat(),
    'investable_exchanges': INVESTABLE_EXCHANGES,
    'common_issuer_types': COMMON_ISSUER_TYPES,
    'common_stock_mapping': {
        'SecurityType': 'EQTY',
        'SecuritySubType': 'COM',
        'ShareType': 'NS',
        'USIncFlg': 'Y',
        'IssuerType': COMMON_ISSUER_TYPES,
    },
    'retention_rule': 'complete path for every PERMNO ever mapped to SHRCD 10/11',
    'output_columns': OUTPUT_COLUMNS,
}
config_digest = sha256(
    json.dumps(config_payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()

print(f'Source rows      : {source_meta.num_rows:,}')
print(f'Source columns   : {len(source_arrow_schema.names)}')
print(f'Configuration ID : {config_digest[:16]}')

## 3. Point-in-time classification flags

`is_common_stock_10_11` identifies the economic security class. `is_regular_active` and `is_investable_exchange` describe the status observable on the current daily row. Their conjunction, `is_formation_eligible`, is the only flag that the formation snapshot in notebook 01 may use.

The three exclusion flags are diagnostics, not alternative selection rules. The exact admissibility rule remains the legacy-10/11 mapping above.

In [ ]:
def normalized_code(column: str) -> pl.Expr:
    return (
        pl.col(column)
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_uppercase()
    )


common_stock_expr = (
    (normalized_code('SecurityType') == 'EQTY')
    & (normalized_code('SecuritySubType') == 'COM')
    & (normalized_code('ShareType') == 'NS')
    & (normalized_code('USIncFlg') == 'Y')
    & normalized_code('IssuerType').is_in(COMMON_ISSUER_TYPES)
)

regular_active_expr = (
    (normalized_code('ConditionalType') == 'RW')
    & (normalized_code('TradingStatusFlg') == 'A')
)

investable_exchange_expr = normalized_code('PrimaryExch').is_in(INVESTABLE_EXCHANGES)

adr_diagnostic_expr = (
    normalized_code('SecurityType').is_in(['ADRT', 'ADR'])
    | normalized_code('ShrAdrFlg').is_in(['Y', 'YES', 'ADR'])
)

reit_diagnostic_expr = (
    normalized_code('SecuritySubType').is_in(['REIT', 'REITS'])
    | normalized_code('IssuerType').is_in(['REIT', 'REITS'])
)

fund_diagnostic_expr = (
    normalized_code('SecurityType').is_in(['FUND', 'ETF', 'ETMF'])
    | normalized_code('SecuritySubType').is_in([
        'CEF', 'CETF', 'ETF', 'ETMF', 'FUND', 'MF', 'OEF', 'UIT',
    ])
)

source_lf = (
    pl.scan_parquet(SOURCE, low_memory=True, rechunk=False)
    .filter(pl.col('DlyCalDt').dt.date().is_between(START_DATE, END_DATE, closed='both'))
    .select(SOURCE_COLUMNS)
    .with_columns(
        common_stock_expr.alias('is_common_stock_10_11'),
        regular_active_expr.alias('is_regular_active'),
        investable_exchange_expr.alias('is_investable_exchange'),
        adr_diagnostic_expr.alias('is_adr_diagnostic'),
        reit_diagnostic_expr.alias('is_reit_diagnostic'),
        fund_diagnostic_expr.alias('is_fund_diagnostic'),
    )
    .with_columns(
        (
            pl.col('is_common_stock_10_11')
            & pl.col('is_regular_active')
            & pl.col('is_investable_exchange')
        ).alias('is_formation_eligible')
    )
    .select(OUTPUT_COLUMNS)
)

print('Lazy source plan constructed. The daily panel has not been materialised.')

## 4. Cache decision

A cached output is accepted only if the manifest carries the same source signature, configuration digest, and declared output schema. `FORCE_RECOMPUTE=True` bypasses this guard. The strong source checksum is computed during a fresh build with bounded memory; it is not recomputed on every reload unless explicitly requested.

In [ ]:
def read_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def sha256_file(path: Path, block_size: int = 8 * 2**20) -> str:
    digest = sha256()
    with path.open('rb') as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


existing_manifest = read_json(MANIFEST)
cache_matches = bool(
    OUTPUT.exists()
    and existing_manifest
    and existing_manifest.get('source_signature') == source_signature
    and existing_manifest.get('config_digest') == config_digest
    and existing_manifest.get('output_columns') == OUTPUT_COLUMNS
)

if cache_matches and VERIFY_SOURCE_SHA256_ON_RELOAD:
    expected_sha = existing_manifest.get('source_sha256')
    cache_matches = bool(expected_sha and sha256_file(SOURCE) == expected_sha)

REBUILD = FORCE_RECOMPUTE or not cache_matches
print('Decision:', 'COMPUTE' if REBUILD else 'RELOAD')

## 5. Streamed build and atomic publication

The first pass reads only the columns required to identify `PERMNO`s that have ever satisfied the ordinary-common-stock mapping. The resulting identifier table is small. The second pass projects the full daily paths of those identifiers directly to a temporary Parquet file.

No global sort or daily-panel deduplication is performed here: both would require substantially more memory. Row identity remains inherited from the immutable RAW contract and is audited downstream.

In [ ]:
source_sha256 = None
n_retained_permnos = None
temporary_output = OUTPUT.with_name(f'.{OUTPUT.name}.tmp.parquet')

if REBUILD:
    print('Pass 1/2 — collecting the small set of ever-common PERMNOs...')
    retained_permnos = (
        source_lf
        .filter(pl.col('is_common_stock_10_11'))
        .select('PERMNO')
        .unique()
        .collect(engine='streaming')
        .sort('PERMNO')
    )
    n_retained_permnos = retained_permnos.height
    assert n_retained_permnos > 0, 'The common-stock mapping selected no PERMNO.'
    assert retained_permnos['PERMNO'].null_count() == 0
    assert retained_permnos['PERMNO'].n_unique() == n_retained_permnos
    print(f'Retained PERMNOs: {n_retained_permnos:,}')

    export_lf = source_lf.join(retained_permnos.lazy(), on='PERMNO', how='semi')

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    if temporary_output.exists():
        temporary_output.unlink()

    print('Pass 2/2 — streaming retained daily paths to temporary Parquet...')
    try:
        export_lf.sink_parquet(
            temporary_output,
            compression=COMPRESSION,
            compression_level=COMPRESSION_LEVEL,
            statistics=True,
            row_group_size=ROW_GROUP_SIZE,
            maintain_order=True,
            mkdir=True,
        )
    except Exception:
        if temporary_output.exists():
            temporary_output.unlink()
        raise

    if COMPUTE_SOURCE_SHA256_ON_BUILD:
        print('Computing source SHA-256 with an 8 MiB bounded-memory buffer...')
        source_sha256 = sha256_file(SOURCE)

    print(f'Temporary output written; validation is required before promotion: {temporary_output}')
else:
    source_sha256 = existing_manifest.get('source_sha256')
    n_retained_permnos = existing_manifest.get('n_retained_permnos')
    print(f'Reloaded: {OUTPUT}')

## 6. Integrity and exclusion assertions

The following checks remain streaming or metadata-only. They verify the exact schema, the PIT logical implications, the complete-path retention rule, the date boundary, and the absence of permanent non-common security identifiers from the compact source.

These assertions validate the source-building layer. The monthly cardinality, breakpoint, lookback, and holding-period assertions remain in the dedicated universe test notebook.

In [ ]:
validation_path = temporary_output if REBUILD else OUTPUT
assert validation_path.exists(), validation_path
output_file = pq.ParquetFile(validation_path)
output_meta = output_file.metadata
output_columns = output_file.schema_arrow.names

assert output_columns == OUTPUT_COLUMNS, 'Output schema or order differs from the frozen contract.'
assert 0 < output_meta.num_rows <= source_meta.num_rows

output_lf = pl.scan_parquet(validation_path, low_memory=True, rechunk=False)

logical_failure = (
    output_lf
    .filter(
        pl.col('is_formation_eligible')
        & ~(
            pl.col('is_common_stock_10_11')
            & pl.col('is_regular_active')
            & pl.col('is_investable_exchange')
        )
    )
    .select(['PERMNO', 'DlyCalDt'])
    .limit(1)
    .collect(engine='streaming')
)
assert logical_failure.is_empty(), 'Formation eligibility violates its declared conjunction.'

excluded_class_failure = (
    output_lf
    .filter(
        pl.col('is_formation_eligible')
        & (
            pl.col('is_adr_diagnostic')
            | pl.col('is_reit_diagnostic')
            | pl.col('is_fund_diagnostic')
        )
    )
    .select(['PERMNO', 'DlyCalDt'])
    .limit(1)
    .collect(engine='streaming')
)
assert excluded_class_failure.is_empty(), 'ADR, REIT, or fund leaked into formation eligibility.'

retention_failure = (
    output_lf
    .group_by('PERMNO')
    .agg(pl.col('is_common_stock_10_11').any().alias('ever_common'))
    .filter(~pl.col('ever_common'))
    .limit(1)
    .collect(engine='streaming')
)
assert retention_failure.is_empty(), 'A retained PERMNO is never classified as common stock.'

summary = output_lf.select(
    pl.len().alias('rows'),
    pl.col('PERMNO').n_unique().alias('permnos'),
    pl.col('DlyCalDt').min().alias('date_min'),
    pl.col('DlyCalDt').max().alias('date_max'),
    pl.col('is_common_stock_10_11').sum().alias('common_stock_rows'),
    pl.col('is_regular_active').sum().alias('regular_active_rows'),
    pl.col('is_formation_eligible').sum().alias('formation_eligible_rows'),
    pl.col('is_adr_diagnostic').sum().alias('adr_diagnostic_rows_retained_path'),
    pl.col('is_reit_diagnostic').sum().alias('reit_diagnostic_rows_retained_path'),
    pl.col('is_fund_diagnostic').sum().alias('fund_diagnostic_rows_retained_path'),
).collect(engine='streaming')

row = summary.row(0, named=True)
assert row['date_min'].date() >= START_DATE
assert row['date_max'].date() <= END_DATE
assert row['formation_eligible_rows'] <= row['common_stock_rows'] <= row['rows']

print('PASS metadata and schema contract')
print('PASS point-in-time eligibility implications')
print('PASS ADR, REIT, and fund formation exclusions')
print('PASS complete-path PERMNO retention contract')
display(summary)

if REBUILD:
    temporary_output.replace(OUTPUT)
    print(f'PASS atomically promoted validated output: {OUTPUT}')

## 7. Signed manifest

The manifest records the immutable input identity, the classification contract, output metadata, and the compact file checksum. It is written only after all assertions pass.

In [ ]:
output_sha256 = sha256_file(OUTPUT)

manifest_payload = {
    'schema_version': config_payload['schema_version'],
    'config_digest': config_digest,
    'source_signature': source_signature,
    'source_sha256': source_sha256,
    'output_path': str(OUTPUT.relative_to(ROOT)),
    'output_size_bytes': OUTPUT.stat().st_size,
    'output_rows': output_meta.num_rows,
    'output_columns': OUTPUT_COLUMNS,
    'output_sha256': output_sha256,
    'n_retained_permnos': int(row['permnos']),
    'audit_summary': {
        key: (value.isoformat() if hasattr(value, 'isoformat') else int(value))
        for key, value in row.items()
    },
    'configuration': config_payload,
}

temporary_manifest = MANIFEST.with_name(f'.{MANIFEST.name}.tmp')
with temporary_manifest.open('w', encoding='utf-8') as handle:
    json.dump(manifest_payload, handle, indent=2, sort_keys=True)
    handle.write('\n')
temporary_manifest.replace(MANIFEST)

print(f'PASS output SHA-256: {output_sha256}')
print(f'Written manifest   : {MANIFEST}')

## 8. Downstream integration contract

Run `tests/00_assert_common_stock_source.ipynb` immediately after this notebook. Only after that independent source audit passes should `01_build_pit_big_small_caps.ipynb` use this output and apply `is_formation_eligible == True` when constructing the investable daily history and the month-end formation snapshot. NYSE breakpoints must therefore be computed from eligible NYSE ordinary common stocks only.

The unfiltered rows retained for the same `PERMNO` must remain available to the daily export builder. They preserve exchange migrations, terminal observations, and Shumway-adjusted delisting returns during the subsequent holding month.

Do not delete the previous essentials projection or invalidate downstream caches until notebook 01 and its assertion notebook pass against this new source.